<a href="https://colab.research.google.com/github/Teniola88/AI-and-Drug-Discovery-Course-assignment/blob/main/QSAR_Part_3_Pubchem_Fingerprint_Calculation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **AI And Biotechnology/Bioinformatics**

## **AI and Drug Discovery Course: QSAR Modeling**
This notebook demonstrates how to collect and preprocess bioactivity data from ChEMBL for QSAR modeling

# **Part 3: Descriptor Calculation**

PaDELPy is a Python wrapper for the PaDEL-Descriptor (molecular descriptor calculation) software.  

It provide the following descriptors/fingerprint:  
* 1444 - 2D Descriptors
* 431 - 3D Descriptors
* 881 bits - PubChem Fingerprints

## **Install PaDELpy**

In [1]:
!pip install padelpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.9/20.9 MB 37.0 MB/s eta 0:00:00


## **Import libraries**

In [4]:
import pandas as pd
import numpy as np
from google.colab import files
uploaded = files.upload()
from padelpy import padeldescriptor

Saving df_lipinski.csv to df_lipinski.csv


## **Load dataset**

In [5]:
df = pd.read_csv('df_lipinski.csv')
df.head()

,molecule_chembl_id,bioactivity_class,pIC50,canonical_smiles,MW,LogP,NumHDonors,NumHAcceptors
0,CHEMBL5821675,active,6.519275,C#CC(=O)N1CC[C@H](C(=O)N(C)[C@H](C(=O)N[C@H]2C...,902.106,5.41630,3.0,10.0
1,CHEMBL5979784,active,7.173925,C#CCNc1cccc(F)c1-c1nc2c(cc1Cl)c(N1C[C@@H](C)N(...,628.152,5.72372,1.0,7.0
2,CHEMBL5930957,active,8.795880,C#CCc1ccc(O)cc1-c1ncc2c(N3CC4CCC(C3)N4)nc(OCC3...,528.632,3.66000,2.0,8.0
3,CHEMBL5969896,inactive,5.277778,C#CCn1c(-c2cccnc2[C@H](C)OC)c2c3cc(ccc31)-c1cc...,898.118,5.87680,2.0,9.0
4,CHEMBL5612217,active,8.701147,C#Cc1c(F)ccc2cc(N)cc(-c3ncc4c(N(C)[C@H]5C[C@@H...,570.619,5.40660,1.0,7.0


In [6]:
data = df[['canonical_smiles', 'molecule_chembl_id']]
data.head()

,canonical_smiles,molecule_chembl_id
0,C#CC(=O)N1CC[C@H](C(=O)N(C)[C@H](C(=O)N[C@H]2C...,CHEMBL5821675
1,C#CCNc1cccc(F)c1-c1nc2c(cc1Cl)c(N1C[C@@H](C)N(...,CHEMBL5979784
2,C#CCc1ccc(O)cc1-c1ncc2c(N3CC4CCC(C3)N4)nc(OCC3...,CHEMBL5930957
3,C#CCn1c(-c2cccnc2[C@H](C)OC)c2c3cc(ccc31)-c1cc...,CHEMBL5969896
4,C#Cc1c(F)ccc2cc(N)cc(-c3ncc4c(N(C)[C@H]5C[C@@H...,CHEMBL5612217


## **Convert to .smi format**

In [7]:
df_smi = data['canonical_smiles'].to_csv('smiles_chembl.smi', index=None, header=None)

In [8]:
! cat smiles_chembl.smi | head

C#CC(=O)N1CC[C@H](C(=O)N(C)[C@H](C(=O)N[C@H]2Cc3cc(O)cc(c3)-c3ccc4c(c3)c(c(-c3cnccc3[C@H](C)OC)n4CC)CC(C)(C)COC(=O)[C@@H]3CCCN(N3)C2=O)C(C)C)C1
C#CCNc1cccc(F)c1-c1nc2c(cc1Cl)c(N1C[C@@H](C)N(C(=O)C=C)C[C@@H]1C)nc(=O)n2-c1c(C)ccnc1C(C)C
C#CCc1ccc(O)cc1-c1ncc2c(N3CC4CCC(C3)N4)nc(OCC34CCCN3CCC4)nc2c1F
C#CCn1c(-c2cccnc2[C@H](C)OC)c2c3cc(ccc31)-c1cccc(c1)C[C@H](NC(=O)[C@H](C(C)C)N(C)C(=O)[C@H]1CCN(C(=O)C=C)C1)C(=O)N1CCC[C@H](N1)C(=O)OCC(C)(C)C2
C#Cc1c(F)ccc2cc(N)cc(-c3ncc4c(N(C)[C@H]5C[C@@H]5F)nc(OCC56CCCN5CC(=C)C6)nc4c3F)c12
C#Cc1c(F)ccc2cc(O)cc(-c3ncc4c(N(C)[C@H]5C[C@@H]5F)nc(OCC56CCCN5CC(=C)C6)nc4c3F)c12
C#Cc1c(F)ccc2cc(O)cc(-c3ncc4c(N(C)[C@H]5C[C@@H]5F)nc(OCC56CCCN5CC(=C)C6)nc4c3F)c12
C#Cc1c(F)ccc2cc(O)cc(-c3ncc4c(N(C)[C@H]5C[C@@H]5F)nc(OC[C@@]56CCCN5CC(=C)C6)nc4c3F)c12
C#Cc1c(F)ccc2cc(O)cc(-c3ncc4c(N(C)[C@H]5C[C@@H]5F)nc(OC[C@@]56CCCN5C[C@H](F)C6)nc4c3F)c12
C#Cc1c(F)ccc2cc(O)cc(-c3ncc4c(N(C)[C@H]5C[C@@H]5F)nc(OC[C@]56CCCN5CC(=C)C6)nc4c3F)c12


## **Calculate molecular Pubchem Fingerprints using "padeldescriptor" function**


In [9]:
padeldescriptor(mol_dir= "smiles_chembl.smi",
                d_file='pubchem_fingerprints.csv',
                fingerprints = True,
                retainorder= True,
                #removesalt = True, standardizetautomers = True, standardizenitro=True
                )

In [10]:
!ls -lh pubchem_fingerprints.csv

-rw-r--r-- 1 root root 4.2M Feb 25 21:47 pubchem_fingerprints.csv


In [11]:
df_fingerprint = pd.read_csv("pubchem_fingerprints.csv")
df_fingerprint.head()

,Name,PubchemFP0,PubchemFP1,PubchemFP2,PubchemFP3,PubchemFP4,PubchemFP5,PubchemFP6,PubchemFP7,PubchemFP8,...,PubchemFP871,PubchemFP872,PubchemFP873,PubchemFP874,PubchemFP875,PubchemFP876,PubchemFP877,PubchemFP878,PubchemFP879,PubchemFP880
0,AUTOGEN_smiles_chembl_1,1,1,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,AUTOGEN_smiles_chembl_2,1,1,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,AUTOGEN_smiles_chembl_3,1,1,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,AUTOGEN_smiles_chembl_4,1,1,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,AUTOGEN_smiles_chembl_5,1,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## **Prepare Dataset for ML**

In [12]:
df.head()

,molecule_chembl_id,bioactivity_class,pIC50,canonical_smiles,MW,LogP,NumHDonors,NumHAcceptors
0,CHEMBL5821675,active,6.519275,C#CC(=O)N1CC[C@H](C(=O)N(C)[C@H](C(=O)N[C@H]2C...,902.106,5.41630,3.0,10.0
1,CHEMBL5979784,active,7.173925,C#CCNc1cccc(F)c1-c1nc2c(cc1Cl)c(N1C[C@@H](C)N(...,628.152,5.72372,1.0,7.0
2,CHEMBL5930957,active,8.795880,C#CCc1ccc(O)cc1-c1ncc2c(N3CC4CCC(C3)N4)nc(OCC3...,528.632,3.66000,2.0,8.0
3,CHEMBL5969896,inactive,5.277778,C#CCn1c(-c2cccnc2[C@H](C)OC)c2c3cc(ccc31)-c1cc...,898.118,5.87680,2.0,9.0
4,CHEMBL5612217,active,8.701147,C#Cc1c(F)ccc2cc(N)cc(-c3ncc4c(N(C)[C@H]5C[C@@H...,570.619,5.40660,1.0,7.0


In [13]:
# Select only the columns we need for ML
meta_cols = df[['molecule_chembl_id', 'bioactivity_class', 'pIC50']]

# Reset index to ensure proper alignment
meta_cols = meta_cols.reset_index(drop=True)
df_fingerprint = df_fingerprint.reset_index(drop=True)

# Combine meta data with fingerprints
combined_df = pd.concat([meta_cols, df_fingerprint.drop(df_fingerprint.columns[0], axis=1)], axis=1)

# Inspect the first few rows
combined_df.head()


,molecule_chembl_id,bioactivity_class,pIC50,PubchemFP0,PubchemFP1,PubchemFP2,PubchemFP3,PubchemFP4,PubchemFP5,PubchemFP6,...,PubchemFP871,PubchemFP872,PubchemFP873,PubchemFP874,PubchemFP875,PubchemFP876,PubchemFP877,PubchemFP878,PubchemFP879,PubchemFP880
0,CHEMBL5821675,active,6.519275,1,1,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,CHEMBL5979784,active,7.173925,1,1,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,CHEMBL5930957,active,8.795880,1,1,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,CHEMBL5969896,inactive,5.277778,1,1,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,CHEMBL5612217,active,8.701147,1,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## **Save and download the dataset**

In [14]:
# Save as CSV
combined_df.to_csv("QSAR_dataset.csv", index=False)
print("Combined dataset saved as QSAR_dataset.csv")

# Download file in Colab
files.download("QSAR_dataset.csv")

Combined dataset saved as QSAR_dataset.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# **Calculate other fingerprints**

## **Download xml Files from Github**

In [15]:
!wget https://github.com/AI-Biotechnology-Bioinformatics/Drug_Discovery_AI_Course_2026/raw/main/padel_descriptors_xml.zip

--2026-02-25 21:51:11--  https://github.com/AI-Biotechnology-Bioinformatics/Drug_Discovery_AI_Course_2026/raw/main/padel_descriptors_xml.zip
Resolving github.com (github.com)... 140.82.114.4
Connecting to github.com (github.com)|140.82.114.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/AI-Biotechnology-Bioinformatics/Drug_Discovery_AI_Course_2026/main/padel_descriptors_xml.zip [following]
--2026-02-25 21:51:11--  https://raw.githubusercontent.com/AI-Biotechnology-Bioinformatics/Drug_Discovery_AI_Course_2026/main/padel_descriptors_xml.zip
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 10871 (11K) [application/zip]
Saving to: ‘padel_descriptors_xml.zip’

padel_descriptors_x 100%[=============

## **Unzip all files**

In [16]:
!unzip padel_descriptors_xml.zip

Archive:  padel_descriptors_xml.zip
  inflating: AtomPairs2DFingerprintCount.xml  
  inflating: AtomPairs2DFingerprinter.xml  
  inflating: EStateFingerprinter.xml  
  inflating: ExtendedFingerprinter.xml  
  inflating: Fingerprinter.xml       
  inflating: GraphOnlyFingerprinter.xml  
  inflating: KlekotaRothFingerprintCount.xml  
  inflating: KlekotaRothFingerprinter.xml  
  inflating: MACCSFingerprinter.xml  
  inflating: PubchemFingerprinter.xml  
  inflating: SubstructureFingerprintCount.xml  
  inflating: SubstructureFingerprinter.xml  


## **Calculate Fingerprints**

In [17]:
# Specify the XML file for SubstructureFingerprinter directly
Substruc_fp = "SubstructureFingerprinter.xml"

# Calculate Substructure fingerprints
padeldescriptor(
    mol_dir='smiles_chembl.smi',
    d_file='Substructure_fingerprints.csv',
    fingerprints=True,
    descriptortypes= Substruc_fp,
    retainorder=True
    # removesalt=True, standardizetautomers=True
)

In [18]:
import pandas as pd
fp_df = pd.read_csv('Substructure_fingerprints.csv')
num_fingerprints = fp_df.shape[1] - 1  # subtract 'Name' column
print(f"Number of fingerprint bits generated: {num_fingerprints}")

Number of fingerprint bits generated: 307
